In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor

PROC, GRID = Path("../data/processed"), 0.25
T0, T1 = 2013 * 12 + 0, 2026 * 12 + 9      # month index t = year*12 + (month-1): Jan 2013 .. Oct 2026
REGION = "algeria"                          # we run the other regions later
OUTAGE = ["2017-12", "2018-10", "2022-04", "2022-07", "2022-08", "2024-01", "2024-02", "2024-06",
          "2024-07", "2024-11", "2025-12", "2026-01", "2026-03", "2026-04", "2026-07"] if REGION == "algeria" else []

def fit_loglog_smear(t):
    x, y = np.log(t.modis + 1), np.log(t.viirs + 1)
    b, a = np.polyfit(x, y, 1)
    return b, a, np.mean(np.exp(y - (a + b * x)))

def pred_loglog_smear(p, m):
    b, a, smear = p
    return smear * np.exp(a + b * np.log(m + 1)) - 1

# 1. Table with one row per (square, month): MODIS count, VIIRS count, zeros filled in.
#    Why: the model must also learn the cases where nothing was seen.
def prep(g, name):
    t = g.year.astype("int64") * 12 + g.month.astype("int64") - 1
    g = g.assign(t=t, iy=np.floor(g.lat / GRID).astype("int64"), ix=np.floor(g.lon / GRID).astype("int64"))
    return g.set_index(["t", "iy", "ix"])["count"].rename(name)

out_t = [int(m[:4]) * 12 + int(m[5:7]) - 1 for m in OUTAGE]
months = np.setdiff1d(np.arange(T0, T1 + 1), out_t)
m_ = prep(pd.read_parquet(PROC / f"{REGION}_grid_m.parquet"), "modis")
v_ = prep(pd.read_parquet(PROC / f"{REGION}_grid_v.parquet"), "viirs")
m_, v_ = [s[s.index.get_level_values("t").isin(months)] for s in (m_, v_)]
cells = pd.concat([m_.reset_index()[["iy", "ix"]], v_.reset_index()[["iy", "ix"]]]).drop_duplicates()
cube = cells.merge(pd.DataFrame({"t": months}), how="cross")
cube = cube.join(m_, on=["t", "iy", "ix"]).join(v_, on=["t", "iy", "ix"])
cube[["modis", "viirs"]] = cube[["modis", "viirs"]].fillna(0)

# 2. Inputs, all from MODIS, so they will also exist for 2003-2012.
mod = cube.set_index(["t", "iy", "ix"])["modis"]
nb = np.zeros(len(cube))
for dy in (-1, 0, 1):
    for dx in (-1, 0, 1):
        if (dy, dx) != (0, 0):
            key = pd.MultiIndex.from_arrays([cube.t, cube.iy + dy, cube.ix + dx])
            nb += mod.reindex(key).fillna(0).to_numpy()
cube["modis_nb"] = nb                                                          # MODIS in the 8 neighbours
cube["cell_mean"] = cube.groupby(["iy", "ix"])["modis"].transform("mean")      # the square's usual MODIS level
cube["year"], cube["month"] = cube.t // 12, cube.t % 12 + 1
FEATS = ["modis", "modis_nb", "cell_mean", "month", "iy", "ix"]
tot = cube.groupby("t")[["modis", "viirs"]].sum()
print(len(cube), "square-months |", len(cells), "squares |", len(months), "months")

# 3. Leave-one-year-out: each year is predicted by a model that never saw it.
#    ML predicts each square directly. The baseline predicts the monthly total, then uses the share trick.
cube["ml"], cube["base"] = np.nan, np.nan
for y in sorted(cube.year.unique()):
    tr, te = cube.year != y, cube.year == y
    mdl = HistGradientBoostingRegressor(loss="poisson", max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                        min_samples_leaf=50, l2_regularization=1.0, random_state=0)
    mdl.fit(cube.loc[tr, FEATS], cube.loc[tr, "viirs"])
    cube.loc[te, "ml"] = mdl.predict(cube.loc[te, FEATS])

    p = fit_loglog_smear(tot[tot.index // 12 != y])
    tte = tot[tot.index // 12 == y]
    share = cube.loc[te, "modis"] / cube.loc[te, "t"].map(tte.modis)
    cube.loc[te, "base"] = (share * cube.loc[te, "t"].map(pred_loglog_smear(p, tte.modis))).fillna(0)

# 4. Scores: monthly totals (what you saw before) and individual squares (the map).
res = cube.groupby("t")[["viirs", "ml", "base"]].sum()
res = res[res.viirs > 0]
def score(col):
    rel = (res.viirs - res[col]).abs() / res.viirs
    return {"median monthly-total error": f"{rel.median():.1%}",
            "total bias": f"{res[col].sum() / res.viirs.sum() - 1:+.1%}",
            "square MAE": round(float((cube.viirs - cube[col]).abs().mean()), 3),
            "square correlation": round(float(np.corrcoef(cube.viirs, cube[col])[0, 1]), 3)}
display(pd.DataFrame({"baseline + share trick": score("base"), "ML (per square)": score("ml")}))

yr = res.groupby(res.index // 12)[["viirs", "base", "ml"]].sum().round(0)
yr.index.name = "year"
yr["base err"] = (yr.base / yr.viirs - 1).map("{:+.0%}".format)
yr["ml err"] = (yr.ml / yr.viirs - 1).map("{:+.0%}".format)
display(yr)

# 5. How much of VIIRS sits where MODIS saw nothing that month? The share trick gives those squares 0.
zero = cube.modis == 0
print(f"{cube.viirs[zero].sum() / cube.viirs.sum():.1%} of VIIRS cell-days fall in squares where MODIS saw nothing that month")

59192 square-months | 392 squares | 151 months


,baseline + share trick,ML (per square)
median monthly-total error,16.0%,13.7%
total bias,-3.3%,-1.0%
square MAE,0.83,0.702
square correlation,0.842,0.869


,viirs,base,ml,base err,ml err
year,,,,,
2013,4163.0,3777.0,3931.0,-9%,-6%
2014,8403.0,6927.0,7486.0,-18%,-11%
2015,3846.0,3913.0,3763.0,+2%,-2%
2016,5412.0,5385.0,5668.0,-0%,+5%
2017,6933.0,6583.0,6891.0,-5%,-1%
2018,2116.0,1918.0,2152.0,-9%,+2%
2019,5157.0,4988.0,4961.0,-3%,-4%
2020,7400.0,6782.0,7273.0,-8%,-2%
2021,4760.0,4754.0,4563.0,-0%,-4%


30.9% of VIIRS cell-days fall in squares where MODIS saw nothing that month


In [3]:
TRAIN_FROM = 2016
tr = cube.year >= TRAIN_FROM      # the "known period": 2016-2026
te = cube.year <= 2015            # the "unknown past": 2013, 2014, 2015

# Option B: each square's usual MODIS level, computed from training years only
cm_train = cube[tr].groupby(["iy", "ix"])["modis"].mean().rename("cell_mean_tr")
cube = cube.drop(columns="cell_mean_tr", errors="ignore").join(cm_train, on=["iy", "ix"])

In [4]:
PARAMS = dict(loss="poisson", max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
              min_samples_leaf=50, l2_regularization=1.0, random_state=0)

def fit_predict(feats):
    mdl = HistGradientBoostingRegressor(**PARAMS)
    mdl.fit(cube.loc[tr, feats], cube.loc[tr, "viirs"])
    return mdl.predict(cube.loc[te, feats])

FEATS_A = ["modis", "modis_nb", "cell_mean",    "month", "iy", "ix"]
FEATS_B = ["modis", "modis_nb", "cell_mean_tr", "month", "iy", "ix"]

cube["ml_A"], cube["ml_B"] = np.nan, np.nan
cube.loc[te, "ml_A"] = fit_predict(FEATS_A)
cube.loc[te, "ml_B"] = fit_predict(FEATS_B)

In [11]:
base_tot = cube.loc[te].groupby("t")["base_bt"].transform("sum")   # baseline monthly total
ml_tot   = cube.loc[te].groupby("t")["ml_B"].transform("sum")      # ML monthly total
cube["hyb"] = np.nan
cube.loc[te, "hyb"] = cube.loc[te, "ml_B"] / ml_tot * base_tot     # ML shape, baseline total

In [12]:
# baseline: fit log-log on training-year monthly totals, predict the 2013-2015 totals,
# then split each total by MODIS share
p    = fit_loglog_smear(tot[tot.index // 12 >= TRAIN_FROM])
tte  = tot[tot.index // 12 <= 2015]
share = cube.loc[te, "modis"] / cube.loc[te, "t"].map(tte.modis)
cube["base_bt"] = np.nan
cube.loc[te, "base_bt"] = (share * cube.loc[te, "t"].map(pred_loglog_smear(p, tte.modis))).fillna(0)

d = cube[te]
rows = {}
for name, col in [("baseline", "base_bt"), ("ML A (cell_mean all years)", "ml_A"),
                  ("ML B (cell_mean train only)", "ml_B"), ("hybrid", "hyb")]:
    m = d.groupby("t")[["viirs", col]].sum()
    m = m[m.viirs > 0]
    r = {f"bias {y} (%)": 100 * (d.loc[d.year == y, col].sum() / d.loc[d.year == y, "viirs"].sum() - 1)
         for y in (2013, 2014, 2015)}
    r["total bias (%)"] = 100 * (d[col].sum() / d.viirs.sum() - 1)
    r["median monthly error (%)"] = 100 * ((m.viirs - m[col]).abs() / m.viirs).median()
    r["square MAE"] = (d.viirs - d[col]).abs().mean()
    rows[name] = r
display(pd.DataFrame(rows).round(2))

,baseline,ML A (cell_mean all years),ML B (cell_mean train only),hybrid
bias 2013 (%),-10.86,-8.24,-10.18,-10.86
bias 2014 (%),-20.90,-14.71,-16.58,-20.90
bias 2015 (%),-1.06,-7.82,-9.22,-1.06
total bias (%),-13.70,-11.46,-13.23,-13.70
median monthly error (%),20.41,23.19,24.51,20.41
square MAE,0.86,0.73,0.74,0.76


In [13]:
# ---- Part A: is the baseline's bias a "busy months" problem? ----
d  = cube[te]
mt = d.groupby("t").agg(modis=("modis", "sum"), viirs=("viirs", "sum"), base=("base_bt", "sum"))
mt = mt[mt.viirs > 0].copy()
mt["year"]  = mt.index // 12
mt["level"] = pd.qcut(mt.modis, 3, labels=["quiet", "medium", "busy"])   # grouped by MODIS, not VIIRS

by_level = mt.groupby("level", observed=True).apply(
    lambda g: pd.Series({"months": len(g),
                         "bias (%)": 100 * (g.base.sum() / g.viirs.sum() - 1)}))
display(by_level.round(1))

by_year_level = mt.groupby(["year", "level"], observed=True).apply(
    lambda g: 100 * (g.base.sum() / g.viirs.sum() - 1)).unstack().round(1)
display(by_year_level)

# ---- Part B: does the MODIS-to-VIIRS relationship drift over time? ----
p_all = fit_loglog_smear(tot[tot.index // 12 >= TRAIN_FROM])      # curve fitted on 2016+ only
am = tot[tot.viirs > 0].copy()
am["pred"] = pred_loglog_smear(p_all, am.modis)
am["year"] = am.index // 12
yr = am.groupby("year")[["viirs", "pred"]].sum()
yr["bias (%)"] = 100 * (yr.pred / yr.viirs - 1)
display(yr.round(1))

,months,bias (%)
level,,
quiet,12.0,26.8
medium,12.0,19.2
busy,12.0,-20.2


level,quiet,medium,busy
year,,,
2013,52.8,18.4,-18.6
2014,8.2,20.6,-23.2
2015,19.0,19.2,-13.0


,viirs,pred,bias (%)
year,,,
2013,4163.0,3711.0,-10.9
2014,8403.0,6646.7,-20.9
2015,3846.0,3805.1,-1.1
2016,5412.0,5162.3,-4.6
2017,6933.0,6235.2,-10.1
2018,2116.0,1960.6,-7.3
2019,5157.0,4797.8,-7.0
2020,7400.0,6353.6,-14.1
2021,4760.0,4536.6,-4.7


In [14]:
# ---- the curved version of the baseline ----
def fit_quad_smear(t):
    x, y = np.log(t.modis + 1), np.log(t.viirs + 1)
    c, b, a = np.polyfit(x, y, 2)                       # polyfit returns the highest power first
    smear = np.mean(np.exp(y - (a + b * x + c * x**2)))
    return a, b, c, smear

def pred_quad_smear(p, m):
    a, b, c, smear = p
    x = np.log(m + 1)
    return smear * np.exp(a + b * x + c * x**2) - 1

train_tot = tot[tot.index // 12 >= TRAIN_FROM]          # 2016+ monthly totals
test_tot  = tot[tot.index // 12 <= 2015]                # 2013-2015 monthly totals (hidden from the fit)
pq = fit_quad_smear(train_tot)
print(f"c = {pq[2]:+.4f}   (near 0 means the curve stays straight)")

# ---- check 1: does the test range sit inside the training range? ----
lo, hi = np.log(train_tot.modis + 1).agg(["min", "max"])
tl = np.log(test_tot.modis + 1)
print(f"train log-MODIS range: {lo:.2f} to {hi:.2f} | test: {tl.min():.2f} to {tl.max():.2f} "
      f"| test months outside train range: {int(((tl < lo) | (tl > hi)).sum())} of {len(tl)}")

# ---- split each predicted monthly total across squares by MODIS share (same as before) ----
d = cube.loc[te]
share = d["modis"] / d["t"].map(test_tot.modis)
cube["base_q"] = np.nan
cube.loc[te, "base_q"] = (share * d["t"].map(pred_quad_smear(pq, test_tot.modis))).fillna(0)

# ---- check 2: bias by activity level, straight vs curved ----
d  = cube[te]
mt = d.groupby("t").agg(modis=("modis", "sum"), viirs=("viirs", "sum"),
                        base=("base_bt", "sum"), curved=("base_q", "sum"))
mt = mt[mt.viirs > 0].copy()
mt["year"]  = mt.index // 12
mt["level"] = pd.qcut(mt.modis, 3, labels=["quiet", "medium", "busy"])

def bias(g, col):
    return 100 * (g[col].sum() / g.viirs.sum() - 1)

lv = pd.DataFrame({"straight": mt.groupby("level", observed=True).apply(lambda g: bias(g, "base")),
                   "curved":   mt.groupby("level", observed=True).apply(lambda g: bias(g, "curved"))})
display(lv.round(1))

display(mt.groupby(["year", "level"], observed=True).apply(lambda g: bias(g, "curved")).unstack().round(1))

# ---- check 3: overall scores, straight vs curved ----
def overall(col_sq, col_mt):
    rel = (mt.viirs - mt[col_mt]).abs() / mt.viirs
    return {"total bias (%)": 100 * (mt[col_mt].sum() / mt.viirs.sum() - 1),
            "median monthly error (%)": 100 * rel.median(),
            "square MAE": (d.viirs - d[col_sq]).abs().mean()}
display(pd.DataFrame({"straight": overall("base_bt", "base"),
                      "curved":   overall("base_q", "curved")}).round(2))

# ---- check 4: yearly bias over all years, straight vs curved (does the drift change?) ----
p_line = fit_loglog_smear(train_tot)
am = tot[tot.viirs > 0].copy()
am["year"] = am.index // 12
am["straight"] = pred_loglog_smear(p_line, am.modis)
am["curved"]   = pred_quad_smear(pq, am.modis)
yy = am.groupby("year")[["viirs", "straight", "curved"]].sum()
out = pd.DataFrame({"straight (%)": 100 * (yy.straight / yy.viirs - 1),
                    "curved (%)":   100 * (yy.curved / yy.viirs - 1)})
display(out.round(1))
print("std of yearly bias:  straight", round(out["straight (%)"].std(), 1), "| curved", round(out["curved (%)"].std(), 1))

c = +0.0351   (near 0 means the curve stays straight)
train log-MODIS range: 0.69 to 7.44 | test: 1.10 to 7.47 | test months outside train range: 1 of 36


,straight,curved
level,,
quiet,26.8,34.5
medium,19.2,11.0
busy,-20.2,-13.0


level,quiet,medium,busy
year,,,
2013,63.0,9.3,-16.1
2014,12.2,10.8,-13.2
2015,43.1,11.3,-7.6


,straight,curved
total bias (%),-13.70,-8.19
median monthly error (%),20.41,17.24
square MAE,0.86,0.84


,straight (%),curved (%)
year,,
2013,-10.9,-8.7
2014,-20.9,-11.6
2015,-1.1,-0.3
2016,-4.6,-1.4
2017,-10.1,4.1
2018,-7.3,-10.9
2019,-7.0,-4.3
2020,-14.1,-3.3
2021,-4.7,4.7


std of yearly bias:  straight 13.2 | curved 12.2


In [15]:
# uses mt from the last cell: one row per test month, with modis, viirs, base (straight), curved, level
mt["abs_err_curved"]   = (mt.curved - mt.viirs).abs()
mt["abs_err_straight"] = (mt.base - mt.viirs).abs()

g = mt.groupby("level", observed=True)
summary = pd.DataFrame({
    "months":                 g.size(),
    "median VIIRS / month":   g.viirs.median(),
    "total VIIRS":            g.viirs.sum(),
    "total curved":           g.curved.sum(),
    "net error (curved)":     g.curved.sum() - g.viirs.sum(),
    "abs error (straight)":   g.abs_err_straight.sum(),
    "abs error (curved)":     g.abs_err_curved.sum(),
})
summary["share of curved abs error (%)"] = 100 * summary["abs error (curved)"] / summary["abs error (curved)"].sum()
summary["share of all VIIRS (%)"]        = 100 * summary["total VIIRS"] / summary["total VIIRS"].sum()
display(summary.round(1).T)

# the quiet months one by one
q = mt[mt.level == "quiet"][["year", "modis", "viirs", "curved"]].copy()
q["diff"] = q.curved - q.viirs
q.index.name = "t"
display(q.round(1))

level,quiet,medium,busy
months,12.0,12.0,12.0
median VIIRS / month,58.0,141.0,1030.0
total VIIRS,712.0,1862.0,13838.0
total curved,957.4,2066.0,12044.5
net error (curved),245.4,204.0,-1793.5
abs error (straight),232.3,447.2,3123.4
abs error (curved),279.9,428.4,1988.1
share of curved abs error (%),10.4,15.9,73.7
share of all VIIRS (%),4.3,11.3,84.3


,year,modis,viirs,curved,diff
t,,,,,
24156,2013,2.0,35.0,50.6,15.6
24157,2013,5.0,22.0,68.8,46.8
24158,2013,7.0,43.0,78.9,35.9
24159,2013,10.0,68.0,92.4,24.4
24160,2013,6.0,56.0,74.0,18.0
24167,2013,13.0,64.0,104.7,40.7
24168,2014,5.0,60.0,68.8,8.8
24169,2014,14.0,66.0,108.6,42.6
24170,2014,9.0,89.0,88.1,-0.9


In [ ]:
## Hackathon Day

In [16]:
# ML decides the split across squares, the curved baseline decides each month's total
d  = cube.loc[te]
cur_tot = d.groupby("t")["base_q"].transform("sum")     # curved baseline monthly total
ml_tot  = d.groupby("t")["ml_B"].transform("sum")       # ML monthly total
cube["hyb_q"] = np.nan
cube.loc[te, "hyb_q"] = (d["ml_B"] / ml_tot * cur_tot).fillna(0)

d = cube.loc[te]
rows = {}
for name, col in [("baseline straight", "base_bt"), ("baseline curved", "base_q"),
                  ("ML B", "ml_B"), ("hybrid straight", "hyb"), ("hybrid curved", "hyb_q")]:
    m = d.groupby("t")[["viirs", col]].sum()
    m = m[m.viirs > 0]
    r = {f"bias {y} (%)": 100 * (d.loc[d.year == y, col].sum() / d.loc[d.year == y, "viirs"].sum() - 1)
         for y in (2013, 2014, 2015)}
    r["total bias (%)"] = 100 * (d[col].sum() / d.viirs.sum() - 1)
    r["median monthly error (%)"] = 100 * ((m.viirs - m[col]).abs() / m.viirs).median()
    r["square MAE"] = (d.viirs - d[col]).abs().mean()
    rows[name] = r
display(pd.DataFrame(rows).round(2))

,baseline straight,baseline curved,ML B,hybrid straight,hybrid curved
bias 2013 (%),-10.86,-8.73,-10.18,-10.86,-8.73
bias 2014 (%),-20.90,-11.55,-16.58,-20.90,-11.55
bias 2015 (%),-1.06,-0.26,-9.22,-1.06,-0.26
total bias (%),-13.70,-8.19,-13.23,-13.70,-8.19
median monthly error (%),20.41,17.24,24.51,20.41,17.24
square MAE,0.86,0.84,0.74,0.76,0.73


In [17]:
# yearly bias of the curved fit over all years (curve fitted on 2016+)
am = tot[tot.viirs > 0].copy()
am["year"] = am.index // 12
am["pred"] = pred_quad_smear(pq, am.modis)
yb = 100 * (am.groupby("year").pred.sum() / am.groupby("year").viirs.sum() - 1)

# (1) is there a correction to learn? compare the known period with the hidden one
print(f"mean yearly bias 2016+: {yb[yb.index >= TRAIN_FROM].mean():+.1f}% | "
      f"2013-2015: {yb[yb.index <= 2015].mean():+.1f}%")

# (2) does a +/- band built from the OTHER years contain each year's bias? (leave-one-year-out)
rows = []
for k in (1.0, 1.5, 2.0):
    hit = 0
    for y in yb.index:
        o = yb.drop(y)
        hit += (o.mean() - k * o.std() <= yb[y] <= o.mean() + k * o.std())
    rows.append({"band = mean +/- k*std, k": k, "half-width (%)": round(k * yb.std(), 1),
                 "years inside": f"{hit}/{len(yb)}"})
display(pd.DataFrame(rows))

# (3) the band for a single month (what the calendar shows): pred / true on the hidden months
mm = cube.loc[te].groupby("t")[["viirs", "hyb_q"]].sum()
mm = mm[mm.viirs > 0]
ratio = mm.hyb_q / mm.viirs
print("monthly pred/true on 2013-2015, 10% / 50% / 90% quantiles:",
      ratio.quantile([0.1, 0.5, 0.9]).round(2).tolist())

mean yearly bias 2016+: +3.3% | 2013-2015: -6.8%


,"band = mean +/- k*std, k",half-width (%),years inside
0,1.0,12.2,8/14
1,1.5,18.3,12/14
2,2.0,24.4,13/14


monthly pred/true on 2013-2015, 10% / 50% / 90% quantiles: [0.83, 1.11, 1.74]


In [18]:
def run(feats, **over):
    mdl = HistGradientBoostingRegressor(**{**PARAMS, **over})
    mdl.fit(cube.loc[tr, feats], cube.loc[tr, "viirs"])
    out = {}
    for name, mask in (("train", tr), ("test", te)):
        dd = cube.loc[mask, ["t", "viirs"]].copy()
        dd["p"] = mdl.predict(cube.loc[mask, feats])
        m = dd.groupby("t")[["viirs", "p"]].sum()
        m = m[m.viirs > 0]
        out[f"{name} square MAE"] = (dd.viirs - dd.p).abs().mean()
        out[f"{name} monthly err (%)"] = 100 * ((m.viirs - m.p).abs() / m.viirs).median()
        out[f"{name} total bias (%)"] = 100 * (dd.p.sum() / dd.viirs.sum() - 1)
    return out

variants = {
    "full (B)":                 run(FEATS_B),
    "no iy, ix":                run(["modis", "modis_nb", "cell_mean_tr", "month"]),
    "no iy, ix, cell_mean":     run(["modis", "modis_nb", "month"]),
    "small trees (full feats)": run(FEATS_B, max_leaf_nodes=5, min_samples_leaf=200, max_iter=150),
}
display(pd.DataFrame(variants).round(2))

,full (B),"no iy, ix","no iy, ix, cell_mean",small trees (full feats)
train square MAE,0.64,0.67,0.79,0.77
train monthly err (%),11.32,10.62,14.20,15.50
train total bias (%),-0.16,-0.19,0.29,-0.30
test square MAE,0.74,0.77,0.83,0.83
test monthly err (%),24.51,21.90,21.25,24.48
test total bias (%),-13.23,-12.07,-6.25,-12.61


In [19]:
months_pre = np.arange(2003 * 12, 2013 * 12)

m_all = prep(pd.read_parquet(PROC / f"{REGION}_grid_m.parquet"), "modis")
mp = m_all[m_all.index.get_level_values("t").isin(months_pre)]

# (1) coverage: regional MODIS total per month, 2003-2012
tot_pre = mp.groupby("t").sum().reindex(months_pre, fill_value=0)
cov = pd.DataFrame({"year": months_pre // 12, "month": months_pre % 12 + 1, "modis": tot_pre.to_numpy()})
display(cov.pivot(index="year", columns="month", values="modis").astype(int))
print("for comparison, 2013+ monthly MODIS total: min", int(tot.modis.min()),
      "| median", int(tot.modis.median()), "| max", int(tot.modis.max()))

# (2) the same square-month table as before, for 2003-2012 (MODIS only)
cube_pre = cells[["iy", "ix"]].merge(pd.DataFrame({"t": months_pre}), how="cross")
cube_pre = cube_pre.join(mp, on=["t", "iy", "ix"])
cube_pre["modis"] = cube_pre["modis"].fillna(0)
print(f"MODIS cell-days outside our {len(cells)} squares (dropped): "
      f"{mp.sum() - cube_pre.modis.sum():.0f} of {mp.sum():.0f}")

mod = cube_pre.set_index(["t", "iy", "ix"])["modis"]
nb = np.zeros(len(cube_pre))
for dy in (-1, 0, 1):
    for dx in (-1, 0, 1):
        if (dy, dx) != (0, 0):
            key = pd.MultiIndex.from_arrays([cube_pre.t, cube_pre.iy + dy, cube_pre.ix + dx])
            nb += mod.reindex(key).fillna(0).to_numpy()
cube_pre["modis_nb"] = nb
cube_pre["cell_mean"] = cube_pre.join(cube.groupby(["iy", "ix"])["modis"].mean().rename("cm"),
                                      on=["iy", "ix"])["cm"].to_numpy()   # usual level from 2013+ only
cube_pre["year"], cube_pre["month"] = cube_pre.t // 12, cube_pre.t % 12 + 1
print(len(cube_pre), "square-months for 2003-2012")

month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
2003,2,0,10,11,7,76,632,856,67,30,2,1
2004,3,2,3,7,4,42,352,754,452,564,15,4
2005,5,3,29,13,44,196,1008,957,302,86,70,2
2006,2,2,13,14,10,52,615,373,230,975,122,8
2007,7,4,13,1,13,231,780,3059,119,14,6,10
2008,4,13,14,28,14,52,283,1084,588,16,20,0
2009,0,1,10,2,17,127,1614,774,140,95,50,6
2010,10,13,12,9,30,103,716,803,667,250,8,12
2011,8,8,6,8,3,28,222,695,754,94,10,14


for comparison, 2013+ monthly MODIS total: min 1 | median 31 | max 1762
MODIS cell-days outside our 392 squares (dropped): 6 of 30196
47040 square-months for 2003-2012


In [20]:
RAW = Path("../data/raw") / REGION
for f in ["modis_archive.csv", "modis_nrt.csv", "viirs_archive.csv", "viirs_nrt.csv"]:
    p = RAW / f
    if not p.exists():
        print(f, "missing"); continue
    head = pd.read_csv(p, nrows=3)
    n, yrs = 0, {}
    for ch in pd.read_csv(p, usecols=["acq_date"], chunksize=500_000):
        n += len(ch)
        for y, c in ch.acq_date.str[:4].value_counts().items():
            yrs[y] = yrs.get(y, 0) + c
    print(f"\n{f}: {n:,} rows | columns: {list(head.columns)}")
    print({y: yrs[y] for y in sorted(yrs)})


modis_archive.csv: 231,403 rows | columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']
{'2000': 425, '2001': 4311, '2002': 6058, '2003': 9123, '2004': 8756, '2005': 9249, '2006': 10992, '2007': 13146, '2008': 10907, '2009': 10424, '2010': 10582, '2011': 8732, '2012': 15925, '2013': 9684, '2014': 11367, '2015': 9165, '2016': 10102, '2017': 10511, '2018': 6801, '2019': 9062, '2020': 11545, '2021': 7753, '2022': 5911, '2023': 5307, '2024': 4505, '2025': 5123, '2026': 5937}

modis_nrt.csv: 960 rows | columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight']
{'2026': 960}

viirs_archive.csv: 1,374,116 rows | columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version',

In [21]:
sat = {}
for ch in pd.read_csv(RAW / "modis_archive.csv", usecols=["acq_date", "satellite", "daynight"], chunksize=500_000):
    ch["year"] = ch.acq_date.str[:4]
    for k, c in ch.groupby(["year", "satellite"]).size().items():
        sat[k] = sat.get(k, 0) + c
display(pd.Series(sat).unstack(fill_value=0).loc["2000":"2005"])

,Aqua,Terra
2000,0,425
2001,0,4311
2002,2003,4055
2003,4502,4621
2004,4297,4459
2005,4479,4770


In [22]:
def region_tot(region):
    m = prep(pd.read_parquet(PROC / f"{region}_grid_m.parquet"), "modis").groupby(level="t").sum()
    v = prep(pd.read_parquet(PROC / f"{region}_grid_v.parquet"), "viirs").groupby(level="t").sum()
    t = pd.DataFrame({"modis": m, "viirs": v}).reindex(range(T0, T1 + 1)).fillna(0)
    return t.drop(index=out_t, errors="ignore") if region == "algeria" else t

R = {r: region_tot(r) for r in ("algeria", "portugal", "zambia")}

# (1) does the yearly bias pattern repeat across regions? (straight fit on each region's 2016+)
yb_r = {}
for r, t in R.items():
    p = fit_loglog_smear(t[t.index // 12 >= TRAIN_FROM])
    a = t[t.viirs > 0].copy(); a["year"] = a.index // 12
    a["pred"] = pred_loglog_smear(p, a.modis)
    yb_r[r] = 100 * (a.groupby("year").pred.sum() / a.groupby("year").viirs.sum() - 1)
yb_r = pd.DataFrame(yb_r)
display(yb_r.round(1))
print("correlation of yearly bias between regions:"); display(yb_r.corr().round(2))

# (2) predict Algeria's hidden 2013-2015 totals three ways (log-log, no smearing, for a fair comparison)
def design(t, region):
    x = np.log(t.modis + 1).to_numpy()
    onehot = np.column_stack([(region == r) * 1.0 for r in R])
    return x, onehot
rows = {}
tr_parts = {r: t[(t.index // 12 >= TRAIN_FROM) & (t.viirs > 0)] for r, t in R.items()}
te_alg = R["algeria"][(R["algeria"].index // 12 <= 2015) & (R["algeria"].viirs > 0)]
y_te = te_alg.viirs.to_numpy()

def score(pred):
    return {"median monthly err (%)": 100 * np.median(np.abs(y_te - pred) / y_te),
            "total bias (%)": 100 * (pred.sum() / y_te.sum() - 1)}

# a) Algeria only
x, y = np.log(tr_parts["algeria"].modis + 1), np.log(tr_parts["algeria"].viirs + 1)
b, a = np.polyfit(x, y, 1)
rows["Algeria only"] = score(np.exp(a + b * np.log(te_alg.modis + 1)) - 1)

# b) all regions pooled, one line
X = np.concatenate([np.log(t.modis + 1) for t in tr_parts.values()])
Y = np.concatenate([np.log(t.viirs + 1) for t in tr_parts.values()])
b, a = np.polyfit(X, Y, 1)
rows["pooled, no region term"] = score(np.exp(a + b * np.log(te_alg.modis + 1)) - 1)

# c) all regions pooled, shared slope, one intercept per region
names = list(R)
A = np.column_stack([np.concatenate([np.full(len(t), r == n) for r, t in tr_parts.items()]) * 1.0 for n in names] + [X])
coef = np.linalg.lstsq(A, Y, rcond=None)[0]
rows["pooled, region intercepts"] = score(np.exp(coef[names.index("algeria")] + coef[-1] * np.log(te_alg.modis + 1)) - 1)
display(pd.DataFrame(rows).round(1))

,algeria,portugal,zambia
year,,,
2013,-10.9,-18.9,5.1
2014,-20.9,-5.2,4.9
2015,-1.1,-9.0,5.0
2016,-4.6,-18.5,4.8
2017,-10.1,-16.2,5.4
2018,-7.3,4.2,1.7
2019,-7.0,1.4,2.0
2020,-14.1,15.3,-4.9
2021,-4.7,7.6,1.6


correlation of yearly bias between regions:


,algeria,portugal,zambia
algeria,1.00,-0.12,0.19
portugal,-0.12,1.00,-0.56
zambia,0.19,-0.56,1.00


,Algeria only,"pooled, no region term","pooled, region intercepts"
median monthly err (%),18.7,23.5,28.7
total bias (%),-16.5,13.1,27.9


In [23]:
def ll_fit(parts):
    X = np.concatenate([np.log(t.modis + 1) for t in parts])
    Y = np.concatenate([np.log(t.viirs + 1) for t in parts])
    b, a = np.polyfit(X, Y, 1)
    return a, b

def stats(t, pred):
    rel = np.abs(t.viirs - pred) / t.viirs
    return {"median monthly err (%)": 100 * rel.median(),
            "total bias (%)": 100 * (pred.sum() / t.viirs.sum() - 1)}

rows = {}
for held in R:
    t = R[held][R[held].viirs > 0]
    a, b = ll_fit([R[r][R[r].viirs > 0] for r in R if r != held])
    rows[f"{held}: pooled on the other two"] = stats(t, np.exp(a + b * np.log(t.modis + 1)) - 1)
    a, b = ll_fit([t])
    rows[f"{held}: own fit (in-sample)"] = stats(t, np.exp(a + b * np.log(t.modis + 1)) - 1)
display(pd.DataFrame(rows).T.round(1))

,median monthly err (%),total bias (%)
algeria: pooled on the other two,38.0,9.8
algeria: own fit (in-sample),15.0,-6.6
portugal: pooled on the other two,150.7,162.9
portugal: own fit (in-sample),20.5,-8.1
zambia: pooled on the other two,75.0,-82.5
zambia: own fit (in-sample),14.1,-1.3


In [24]:
t = R["algeria"][R["algeria"].viirs > 0].copy()
t["year"] = t.index // 12
display((t.groupby("year").viirs.sum() / t.groupby("year").modis.sum()).round(2).to_frame("VIIRS/MODIS"))

for name, mask in [("2013-2021", t.year <= 2021), ("2022-2026", t.year >= 2022),
                   ("2013-2018", t.year <= 2018), ("2019-2026", t.year >= 2019)]:
    s = t[mask]
    p = fit_loglog_smear(s)
    print(f"{name}: {len(s)} months | exponent b = {p[0]:.2f} | VIIRS/MODIS = {s.viirs.sum() / s.modis.sum():.2f}")

,VIIRS/MODIS
year,
2013,2.92
2014,2.39
2015,2.77
2016,2.45
2017,1.96
2018,4.65
2019,2.60
2020,2.18
2021,2.21


2013-2021: 106 months | exponent b = 0.67 | VIIRS/MODIS = 2.40
2022-2026: 45 months | exponent b = 0.58 | VIIRS/MODIS = 2.76
2013-2018: 70 months | exponent b = 0.70 | VIIRS/MODIS = 2.46
2019-2026: 81 months | exponent b = 0.61 | VIIRS/MODIS = 2.46


In [25]:
from scipy.spatial import cKDTree

LAT0 = 35.6
KX, KY = 111.32 * np.cos(np.radians(LAT0)), 110.57          # km per degree, near the middle of the crop

def load(fname, conf_ok):
    df = pd.read_csv(RAW / fname, usecols=["latitude", "longitude", "acq_date", "satellite", "confidence", "type"])
    df = df[(df.latitude.between(34.0, 37.2)) & (df.longitude.between(-2.3, 8.7))]
    df = df[df.acq_date.between("2013-01-01", "2025-12-31")]
    df = df[conf_ok(df.confidence)]
    return df

modis = load("modis_archive.csv", lambda c: c.astype(float) >= 30)
viirs = load("viirs_archive.csv", lambda c: c.isin(["n", "h"]))
viirs = viirs[viirs.type == 0]                          # vegetation fires only (the archive has labels)

def pts(df):
    day = (pd.to_datetime(df.acq_date) - pd.Timestamp("2013-01-01")).dt.days.to_numpy()
    return np.column_stack([df.longitude * KX, df.latitude * KY, day * 10_000.0])   # days 10,000 km apart: only same-day pairs can match

pm, pv = pts(modis), pts(viirs)
tm, tv = cKDTree(pm), cKDTree(pv)
print(len(modis), "MODIS rows |", len(viirs), "VIIRS rows")

rows = {}
for R_km in (0.75, 1.5):
    n_v = np.array(tm.query_ball_point(pv, R_km, return_length=True))      # per VIIRS row: MODIS rows nearby
    n_m = np.array(tv.query_ball_point(pm, R_km, return_length=True))      # per MODIS row: VIIRS rows nearby
    for sat, mask in [("all", np.ones(len(modis), bool))] + [(s, (modis.satellite == s).to_numpy()) for s in modis.satellite.unique()]:
        k = n_m[mask]
        rows[f"R={R_km} km, MODIS {sat}"] = {
            "MODIS rows": int(mask.sum()),
            "with 0 VIIRS (%)": 100 * (k == 0).mean(),
            "with exactly 1 (%)": 100 * (k == 1).mean(),
            "with 2-3 (%)": 100 * ((k >= 2) & (k <= 3)).mean(),
            "with 4+ (%)": 100 * (k >= 4).mean(),
            "mean VIIRS per MODIS row": k.mean()}
    rows[f"R={R_km} km, VIIRS side"] = {"VIIRS rows": len(viirs),
                                       "with 0 MODIS (%)": 100 * (n_v == 0).mean()}
display(pd.DataFrame(rows).T.round(1))

36033 MODIS rows | 92500 VIIRS rows


,MODIS rows,with 0 VIIRS (%),with exactly 1 (%),with 2-3 (%),with 4+ (%),mean VIIRS per MODIS row,VIIRS rows,with 0 MODIS (%)
"R=0.75 km, MODIS all",36033.0,64.9,11.2,11.6,12.3,1.2,NaN,NaN
"R=0.75 km, MODIS Aqua",20441.0,57.9,13.7,14.0,14.4,1.4,NaN,NaN
"R=0.75 km, MODIS Terra",15592.0,74.1,7.9,8.4,9.5,0.9,NaN,NaN
"R=0.75 km, VIIRS side",NaN,NaN,NaN,NaN,NaN,NaN,92500.0,68.2
"R=1.5 km, MODIS all",36033.0,51.1,10.1,12.2,26.7,3.4,NaN,NaN
"R=1.5 km, MODIS Aqua",20441.0,40.8,11.9,15.2,32.1,4.1,NaN,NaN
"R=1.5 km, MODIS Terra",15592.0,64.5,7.6,8.4,19.5,2.6,NaN,NaN
"R=1.5 km, VIIRS side",NaN,NaN,NaN,NaN,NaN,NaN,92500.0,56.4


In [26]:
from scipy.spatial import cKDTree

R_KM = 0.75
KX, KY = 111.32 * np.cos(np.radians(35.6)), 110.57

def load(fname, cols, ok):
    df = pd.read_csv(RAW / fname, usecols=cols)
    df = df[df.latitude.between(34.0, 37.2) & df.longitude.between(-2.3, 8.7)]
    df = df[df.acq_date.between("2013-01-01", "2025-12-31")]
    return df[ok(df)].reset_index(drop=True)

M = load("modis_archive.csv",
         ["latitude", "longitude", "brightness", "scan", "track", "acq_date", "acq_time",
          "satellite", "confidence", "frp", "daynight"],
         lambda d: d.confidence.astype(float) >= 30)
V = load("viirs_archive.csv",
         ["latitude", "longitude", "brightness", "acq_date", "confidence", "frp", "type"],
         lambda d: d.confidence.isin(["n", "h"]) & (d.type == 0))

def xyz(df):
    day = (pd.to_datetime(df.acq_date) - pd.Timestamp("2013-01-01")).dt.days.to_numpy()
    return np.column_stack([df.longitude * KX, df.latitude * KY, day * 10_000.0])

# 1. the targets: for each MODIS row, the VIIRS rows within R_KM on the same day
nbrs = cKDTree(xyz(V)).query_ball_point(xyz(M), R_KM)
vfrp = V.frp.to_numpy()
M["n"]     = [len(a) for a in nbrs]                                  # how many VIIRS rows it stands for
M["v_frp"] = [vfrp[a].mean() if len(a) else np.nan for a in nbrs]    # their averaged FRP

# 2. the inputs: only the MODIS row's own properties
d = pd.to_datetime(M.acq_date)
M["year"], M["month"] = d.dt.year, d.dt.month
M["t"]    = M.year * 12 + M.month - 1
M["aqua"] = (M.satellite == "Aqua").astype(int)
M["day"]  = (M.daynight == "D").astype(int)
M["conf"] = M.confidence.astype(float)
M["hour"] = M.acq_time // 100
PHYS = ["brightness", "scan", "track", "frp", "conf", "aqua", "day", "hour", "month"]
SETS = {"physical": PHYS, "physical + lat/lon": PHYS + ["latitude", "longitude"]}
tr, te = M.year >= 2016, M.year <= 2015
print(f"MODIS rows: train {tr.sum()}, test {te.sum()} | with a VIIRS partner: "
      f"{(M.n[tr] > 0).mean():.1%} / {(M.n[te] > 0).mean():.1%}")

# 3. monthly counts: row model vs simple baseline, both in VIIRS rows per month
V["t"] = pd.to_datetime(V.acq_date).dt.year * 12 + pd.to_datetime(V.acq_date).dt.month - 1
vm, mm = V.groupby("t").size(), M.groupby("t").size()
ok  = vm.index.difference(out_t)
trm, tem = ok[ok // 12 >= 2016], ok[ok // 12 <= 2015]
true = vm.loc[tem].to_numpy()

def sc(pred):
    return {"median monthly err (%)": 100 * np.median(np.abs(true - pred) / true),
            "total bias (%)": 100 * (pred.sum() / true.sum() - 1)}

xm = np.log(mm.reindex(ok).fillna(0) + 1); ym = np.log(vm.loc[ok] + 1)
b, a = np.polyfit(xm.loc[trm], ym.loc[trm], 1)
smear = np.mean(np.exp(ym.loc[trm] - (a + b * xm.loc[trm])))
rows = {"baseline (monthly log-log)": sc((smear * np.exp(a + b * xm.loc[tem]) - 1).to_numpy())}

links = M.groupby("t").n.sum()
scale = vm.loc[trm].sum() / links.reindex(trm).fillna(0).sum()       # VIIRS rows per linked row, from training months
P = dict(loss="poisson", max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
         min_samples_leaf=50, l2_regularization=1.0, random_state=0)
for name, feats in SETS.items():
    cnt = HistGradientBoostingRegressor(**P).fit(M.loc[tr, feats], M.loc[tr, "n"])
    pn = pd.Series(cnt.predict(M[feats]), index=M.index).groupby(M.t).sum()
    rows[f"row model, {name}"] = sc((scale * pn.reindex(tem).fillna(0)).to_numpy())
display(pd.DataFrame(rows).T.round(1))

# 4. row level: the averaged VIIRS FRP on matched rows, vs a naive guess
mt, me = M[tr & (M.n > 0)], M[te & (M.n > 0)]
ratio = (mt.v_frp / mt.frp.clip(lower=0.1)).median()
rel = lambda p: 100 * np.median(np.abs(me.v_frp - p) / me.v_frp.clip(lower=0.1))
fr = {"naive (MODIS FRP x training ratio)": rel(ratio * me.frp)}
for name, feats in SETS.items():
    m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                      min_samples_leaf=50, random_state=0).fit(mt[feats], np.log1p(mt.v_frp))
    fr[f"model, {name}"] = rel(np.expm1(m.predict(me[feats])))
print("median relative FRP error on matched test rows (%):")
display(pd.Series(fr).round(1).to_frame("error"))

MODIS rows: train 24589, test 11444 | with a VIIRS partner: 37.1% / 30.7%


,median monthly err (%),total bias (%)
baseline (monthly log-log),74.6,22.7
"row model, physical",44.7,-7.8
"row model, physical + lat/lon",51.9,-22.0


median relative FRP error on matched test rows (%):


,error
naive (MODIS FRP x training ratio),59.5
"model, physical",53.6
"model, physical + lat/lon",51.6


In [27]:
REGS = ["algeria", "portugal", "zambia"]
CROP = {"algeria": (34.0, 37.2, -2.3, 8.7)}              # Portugal and Zambia: whole country file
R_KM, CH = 0.75, 500_000
PHYS = ["brightness", "scan", "track", "frp", "conf", "aqua", "day", "hour", "month"]
P = dict(loss="poisson", max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
         min_samples_leaf=50, l2_regularization=1.0, random_state=0)

def read_filtered(path, cols, keep, crop):
    parts = []
    for ch in pd.read_csv(path, usecols=cols, chunksize=CH):          # chunks: the Zambia VIIRS file is ~1 GB
        if crop:
            ch = ch[ch.latitude.between(*crop[:2]) & ch.longitude.between(*crop[2:])]
        ch = ch[ch.acq_date.between("2013-01-01", "2025-12-31")]
        parts.append(ch[keep(ch)])
    return pd.concat(parts, ignore_index=True)

def build(reg):
    d = Path("../data/raw") / reg
    M = read_filtered(d / "modis_archive.csv",
                      ["latitude", "longitude", "brightness", "scan", "track", "acq_date", "acq_time",
                       "satellite", "confidence", "frp", "daynight"],
                      lambda c: c.confidence.astype(float) >= 30, CROP.get(reg))
    V = read_filtered(d / "viirs_archive.csv",
                      ["latitude", "longitude", "acq_date", "confidence", "type"],
                      lambda c: c.confidence.isin(["n", "h"]) & (c.type == 0), CROP.get(reg))
    kx, ky = 111.32 * np.cos(np.radians(M.latitude.mean())), 110.57
    def xyz(df):
        day = (pd.to_datetime(df.acq_date) - pd.Timestamp("2013-01-01")).dt.days.to_numpy()
        return np.column_stack([df.longitude * kx, df.latitude * ky, day * 10_000.0])
    M["n"] = cKDTree(xyz(V)).query_ball_point(xyz(M), R_KM, return_length=True, workers=-1)
    dt = pd.to_datetime(M.acq_date)
    M["year"], M["month"] = dt.dt.year, dt.dt.month
    M["t"] = M.year * 12 + M.month - 1
    M["aqua"] = (M.satellite == "Aqua").astype(int)
    M["day"]  = (M.daynight == "D").astype(int)
    M["conf"] = M.confidence.astype(float)
    M["hour"] = M.acq_time // 100
    M["region"] = reg
    dv = pd.to_datetime(V.acq_date)
    vm = (dv.dt.year * 12 + dv.dt.month - 1).value_counts().sort_index()      # VIIRS rows per month
    print(reg, len(M), "MODIS rows |", len(V), "VIIRS rows")
    return M, vm

DATA = {r: build(r) for r in REGS}
ALL = pd.concat([DATA[r][0] for r in REGS], ignore_index=True)
for r in REGS:
    ALL["is_" + r] = (ALL.region == r).astype(int)

def months_of(reg):
    vm = DATA[reg][1]
    ok = vm.index.difference(out_t) if reg == "algeria" else vm.index
    return ok[ok // 12 <= 2015], ok[ok // 12 >= 2016]            # hidden months, training months

def score(pred, reg, scale):
    vm, test_m = DATA[reg][1], months_of(reg)[0]
    est  = pred.groupby(ALL.loc[pred.index, "t"]).sum().reindex(test_m).fillna(0) * scale
    true = vm.reindex(test_m)
    return {"median monthly err (%)": 100 * np.median(np.abs(true - est) / true),
            "total bias (%)": 100 * (est.sum() / true.sum() - 1)}

def own_scale(reg):                                               # VIIRS rows per linked MODIS row, training months
    M, vm = DATA[reg]; tr_m = months_of(reg)[1]
    return vm.reindex(tr_m).sum() / M.groupby("t").n.sum().reindex(tr_m).fillna(0).sum()

def fit(rows, feats):
    return HistGradientBoostingRegressor(**P).fit(ALL.loc[rows, feats], ALL.loc[rows, "n"])

# ---- Table 1: Algeria's hidden years, three ways of training ----
alg_te = (ALL.region == "algeria") & (ALL.year <= 2015)
REGF = ["is_" + r for r in REGS]
t1 = {}
for name, rows, feats in [("Algeria only", (ALL.region == "algeria") & (ALL.year >= 2016), PHYS),
                          ("3 countries pooled", ALL.year >= 2016, PHYS),
                          ("3 countries pooled + region flag", ALL.year >= 2016, PHYS + REGF),
                          ("3 countries pooled + lat/lon", ALL.year >= 2016, PHYS + ["latitude", "longitude"])]:
    m = fit(rows, feats)
    t1[name] = score(pd.Series(m.predict(ALL.loc[alg_te, feats]), index=ALL.index[alg_te]), "algeria", own_scale("algeria"))
display(pd.DataFrame(t1).T.round(1))

# ---- Table 2: each country's hidden years, trained on the OTHER two countries (never saw this country) ----
t2 = {}
for held in REGS:
    te_rows = (ALL.region == held) & (ALL.year <= 2015)
    own = fit((ALL.region == held) & (ALL.year >= 2016), PHYS)
    t2[f"{held}: own country, 2016+"] = score(pd.Series(own.predict(ALL.loc[te_rows, PHYS]), index=ALL.index[te_rows]), held, own_scale(held))
    other = [r for r in REGS if r != held]
    rows = ALL.region.isin(other)
    m = fit(rows, PHYS)
    sc = sum(DATA[r][1].reindex(DATA[r][1].index.difference(out_t) if r == "algeria" else DATA[r][1].index).sum() for r in other) / \
         sum(DATA[r][0].n.sum() for r in other)                   # scale from the other two countries only
    t2[f"{held}: other two countries only"] = score(pd.Series(m.predict(ALL.loc[te_rows, PHYS]), index=ALL.index[te_rows]), held, sc)
display(pd.DataFrame(t2).T.round(1))

algeria 36033 MODIS rows | 92500 VIIRS rows
portugal 33294 MODIS rows | 91912 VIIRS rows
zambia 2483273 MODIS rows | 9126182 VIIRS rows


,median monthly err (%),total bias (%)
Algeria only,45.4,-8.0
3 countries pooled,48.1,24.7
3 countries pooled + region flag,28.5,3.4
3 countries pooled + lat/lon,38.3,6.3


,median monthly err (%),total bias (%)
"algeria: own country, 2016+",45.4,-8.0
algeria: other two countries only,261.9,176.4
"portugal: own country, 2016+",60.6,-3.0
portugal: other two countries only,58.9,76.4
"zambia: own country, 2016+",19.0,4.5
zambia: other two countries only,76.8,-42.4


In [28]:
CELL, R_KM = 0.01, 0.75
CROP = {"algeria": (34.0, 37.2, -2.3, 8.7)}
MCOLS = ["latitude", "longitude", "brightness", "scan", "track", "acq_date", "acq_time",
         "satellite", "confidence", "frp", "daynight"]
PHYS = ["brightness", "scan", "track", "frp", "conf", "aqua", "day", "hour", "month"]
P = dict(loss="poisson", max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
         min_samples_leaf=50, l2_regularization=1.0, random_state=0)
D0 = pd.Timestamp("2013-01-01")

def cell_days(df):
    return pd.DataFrame({"cy": np.floor(df.latitude / CELL).astype("int32"),
                         "cx": np.floor(df.longitude / CELL).astype("int32"),
                         "day": (pd.to_datetime(df.acq_date) - D0).dt.days.astype("int32")}).drop_duplicates()

def read_chunks(path, cols, keep, crop, reduce=None):
    out = []
    for ch in pd.read_csv(path, usecols=cols, chunksize=500_000):
        if crop:
            ch = ch[ch.latitude.between(*crop[:2]) & ch.longitude.between(*crop[2:])]
        ch = ch[ch.acq_date.between("2013-01-01", "2025-12-31")]
        ch = ch[keep(ch)]
        out.append(reduce(ch) if reduce else ch)
    return pd.concat(out, ignore_index=True)

def build2(reg):
    d, crop = Path("../data/raw") / reg, CROP.get(reg)
    M = read_chunks(d / "modis_archive.csv", MCOLS, lambda c: c.confidence.astype(float) >= 30, crop)
    V = read_chunks(d / "viirs_archive.csv", ["latitude", "longitude", "acq_date", "confidence", "type"],
                    lambda c: c.confidence.isin(["n", "h"]) & (c.type == 0), crop, reduce=cell_days).drop_duplicates()
    dt = pd.to_datetime(M.acq_date)
    M["year"], M["month"] = dt.dt.year, dt.dt.month
    M["t"] = M.year * 12 + M.month - 1
    M["aqua"], M["day"] = (M.satellite == "Aqua").astype(int), (M.daynight == "D").astype(int)
    M["conf"], M["hour"] = M.confidence.astype(float), M.acq_time // 100
    M["cy"], M["cx"] = np.floor(M.latitude / CELL).astype("int32"), np.floor(M.longitude / CELL).astype("int32")
    M["dayi"] = (dt - D0).dt.days.astype("int32")
    kx, ky = 111.32 * np.cos(np.radians(M.latitude.mean())), 110.57
    pv = np.column_stack([(V.cx + 0.5) * CELL * kx, (V.cy + 0.5) * CELL * ky, V.day * 10_000.0])
    pm = np.column_stack([M.longitude * kx, M.latitude * ky, M.dayi * 10_000.0])
    M["n"] = cKDTree(pv).query_ball_point(pm, R_KM, return_length=True, workers=-1)   # VIIRS cell-days near this row
    vd = D0 + pd.to_timedelta(V.day, unit="D")
    vm  = (vd.dt.year * 12 + vd.dt.month - 1).value_counts().sort_index()              # true VIIRS cell-days per month
    mcd = M.drop_duplicates(["cy", "cx", "dayi"]).groupby("t").size()                  # MODIS cell-days per month
    gap = set(out_t) if reg == "algeria" else set()
    M, vm, mcd = M[~M.t.isin(gap)].copy(), vm[~vm.index.isin(gap)], mcd[~mcd.index.isin(gap)]
    missing = [f"{t // 12}-{t % 12 + 1:02d}" for t in range(2013 * 12, 2026 * 12)
               if t not in gap and t not in vm.index]
    print(f"{reg}: {len(M):,} MODIS rows | {len(V):,} VIIRS cell-days | months with no VIIRS data: {missing}")
    return M, vm, mcd

DATA2 = {r: build2(r) for r in ("algeria", "portugal", "zambia")}
ALL2 = pd.concat([DATA2[r][0].assign(region=r) for r in DATA2], ignore_index=True)
for r in DATA2:
    ALL2["is_" + r] = (ALL2.region == r).astype(int)
FEATS = PHYS + ["is_" + r for r in DATA2]

tr = ALL2.year >= 2016
model = HistGradientBoostingRegressor(**P).fit(ALL2.loc[tr, FEATS], ALL2.loc[tr, "n"])
ALL2["pred"] = model.predict(ALL2[FEATS])

def stats(true, est):
    return {"median monthly err (%)": 100 * np.median(np.abs(true - est) / true),
            "total bias (%)": 100 * (est.sum() / true.sum() - 1)}

rows = {}
for reg, (M, vm, mcd) in DATA2.items():
    test_m, train_m = vm.index[vm.index // 12 <= 2015], vm.index[vm.index // 12 >= 2016]
    true = vm.loc[test_m]
    g = ALL2[ALL2.region == reg].groupby("t")
    scale = vm.loc[train_m].sum() / g.n.sum().reindex(train_m).fillna(0).sum()
    rows[f"{reg}: pooled + region flag"] = stats(true, scale * g.pred.sum().reindex(test_m).fillna(0))
    tt = pd.DataFrame({"modis": mcd.reindex(vm.index).fillna(0), "viirs": vm})
    p = fit_loglog_smear(tt.loc[train_m])
    rows[f"{reg}: log-log baseline"] = stats(true, pred_loglog_smear(p, tt.loc[test_m].modis))
    if reg == "algeria":
        q = fit_quad_smear(tt.loc[train_m])
        rows[f"{reg}: curved baseline"] = stats(true, pred_quad_smear(q, tt.loc[test_m].modis))
display(pd.DataFrame(rows).T.round(1))
print("for reference, the earlier cell-day hybrid on Algeria: 17.2 / -8.2 (its truth is defined slightly differently)")

algeria: 34,862 MODIS rows | 58,217 VIIRS cell-days | months with no VIIRS data: []
portugal: 33,294 MODIS rows | 41,530 VIIRS cell-days | months with no VIIRS data: []
zambia: 2,483,273 MODIS rows | 6,023,854 VIIRS cell-days | months with no VIIRS data: []


,median monthly err (%),total bias (%)
algeria: pooled + region flag,28.0,8.4
algeria: log-log baseline,63.6,18.0
algeria: curved baseline,60.9,19.9
portugal: pooled + region flag,63.0,-19.1
portugal: log-log baseline,15.1,-10.5
zambia: pooled + region flag,21.8,4.5
zambia: log-log baseline,17.7,8.5


for reference, the earlier cell-day hybrid on Algeria: 17.2 / -8.2 (its truth is defined slightly differently)


In [29]:
import json
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss

REG, RAWD, OUT = "algeria", Path("../data/raw") / "algeria", PROC / "export_algeria"
OUT.mkdir(exist_ok=True)
crop = CROP[REG]

def rd(fname, cols, keep, d0, d1):
    parts = []
    for ch in pd.read_csv(RAWD / fname, usecols=cols, chunksize=500_000):
        ch = ch[ch.latitude.between(*crop[:2]) & ch.longitude.between(*crop[2:])]
        ch = ch[ch.acq_date.between(d0, d1)]
        parts.append(ch[keep(ch)])
    return pd.concat(parts, ignore_index=True)

M = rd("modis_archive.csv", MCOLS + ["type"],
       lambda c: (c.confidence.astype(float) >= 30) & (c.type == 0), "2003-01-01", "2025-12-31")
V = rd("viirs_archive.csv", ["latitude", "longitude", "acq_date", "acq_time", "satellite", "confidence", "frp", "type"],
       lambda c: c.confidence.isin(["n", "h"]) & (c.type == 0), "2013-01-01", "2025-12-31")

# ---- cell-day tables (one row per 0.01 degree cell and day) ----
for df in (M, V):
    df["cy"] = np.floor(df.latitude / CELL).astype("int32")
    df["cx"] = np.floor(df.longitude / CELL).astype("int32")
    df["date"] = pd.to_datetime(df.acq_date)
M["conf"], M["aqua"] = M.confidence.astype(float), (M.satellite == "Aqua").astype(int)
M["dayflag"], M["hour"] = (M.daynight == "D").astype(int), M.acq_time // 100
key = ["cy", "cx", "date"]
mcd = M.groupby(key).agg(modis_rows=("frp", "size"), frp_max=("frp", "max"), frp_sum=("frp", "sum"),
                         conf_max=("conf", "max"), bright_max=("brightness", "max"), scan_mean=("scan", "mean"),
                         aqua_any=("aqua", "max"), aqua_share=("aqua", "mean"), day_any=("dayflag", "max"),
                         hour_min=("hour", "min")).reset_index()
vcd = V.groupby(key).agg(viirs_rows=("frp", "size")).reset_index()
cd = mcd.merge(vcd, on=key, how="outer")
cd["modis_rows"] = cd.modis_rows.fillna(0).astype(int)
cd["year"], cd["month"] = cd.date.dt.year, cd.date.dt.month
cd["t"] = cd.year * 12 + cd.month - 1
gap = cd.t.isin(out_t)
known = (cd.date >= "2013-01-01") & ~gap                      # VIIRS truth exists here
cd["viirs_cd"] = np.where(known, (cd.viirs_rows.fillna(0) > 0).astype(float), np.nan)
cd["viirs_rows"] = np.where(known, cd.viirs_rows.fillna(0), np.nan)
FE = ["modis_rows", "frp_max", "frp_sum", "conf_max", "bright_max", "scan_mean",
      "aqua_any", "aqua_share", "day_any", "hour_min", "month"]
lk = cd.modis_rows > 0                                         # cell-days where MODIS saw something
print(f"{lk.sum():,} MODIS cell-days | {int((cd.viirs_cd == 1).sum()):,} VIIRS cell-days | "
      f"{int(((cd.viirs_cd == 1) & lk).sum()):,} of them also have MODIS the same day")

def clf():
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                          min_samples_leaf=50, l2_regularization=1.0, random_state=0)

# ---- rehearsal: train 2016-2025, score the hidden 2013-2015 ----
tr = lk & known & (cd.year >= 2016)
te = lk & known & (cd.year <= 2015)
m_re = clf().fit(cd.loc[tr, FE], cd.loc[tr, "viirs_cd"])
cd["p_re"] = np.nan
cd.loc[lk, "p_re"] = m_re.predict_proba(cd.loc[lk, FE])[:, 1]

tv = cd[cd.viirs_cd == 1].groupby("t").size()                  # true VIIRS cell-days per month
tl = cd[(cd.viirs_cd == 1) & lk].groupby("t").size()           # the part on MODIS cell-days
months_all = sorted(set(cd.t[known]))
tr_months = [t for t in months_all if t // 12 >= 2016]
te_months = [t for t in months_all if t // 12 <= 2015]
scale_tr = tv.reindex(tr_months).sum() / tl.reindex(tr_months).sum()
sp = cd[te].groupby("t").p_re.sum().reindex(te_months).fillna(0)
def mstats(true, est):
    return {"median monthly err (%)": 100 * np.median(np.abs(true - est) / true),
            "total bias (%)": 100 * (est.sum() / true.sum() - 1)}
y, p = cd.loc[te, "viirs_cd"], cd.loc[te, "p_re"]
base = cd.loc[tr, "viirs_cd"].mean()
ev = {"cell-day: AUC": roc_auc_score(y, p), "cell-day: Brier": brier_score_loss(y, p),
      "cell-day: Brier of a constant guess": brier_score_loss(y, np.full(len(y), base)),
      "monthly, linked part only (sum p vs VIIRS on MODIS cell-days)": mstats(tl.reindex(te_months).fillna(0).replace(0, np.nan), sp),
      "monthly, scaled to all VIIRS cell-days": mstats(tv.reindex(te_months), sp * scale_tr),
      "linkage scale (train years)": scale_tr}
display(pd.Series({k: v for k, v in ev.items() if not isinstance(v, dict)}).round(3).to_frame("value"))
display(pd.DataFrame({k: v for k, v in ev.items() if isinstance(v, dict)}).T.round(1))

# ---- final model: train on all known months (2013-2025), predict every MODIS cell-day (incl. 2003-2012) ----
trf = lk & known
m_fin = clf().fit(cd.loc[trf, FE], cd.loc[trf, "viirs_cd"])
cd["p_final"] = np.nan
cd.loc[lk, "p_final"] = m_fin.predict_proba(cd.loc[lk, FE])[:, 1]
scale_fin = tv.reindex(months_all).sum() / tl.reindex(months_all).sum()
cd["p_viirs"] = np.where(cd.year <= 2015, np.where(cd.date < "2013-01-01", cd.p_final, cd.p_re), cd.p_final)
cd.loc[~lk, "p_viirs"] = np.nan
cd["pred_scaled"] = cd.p_viirs * scale_fin
cd["split"] = np.select([gap, cd.date < "2013-01-01", cd.year <= 2015], ["viirs_gap", "reconstructed", "test_hidden"], "train")
print("rows per split:", cd.split.value_counts().to_dict(), "| final linkage scale:", round(scale_fin, 2))

# ---- the five deliverables ----
cd["cell_id"] = cd.cy.astype(str) + "_" + cd.cx.astype(str)
cd["date"] = cd.date.dt.strftime("%Y-%m-%d")
cd[["date", "cell_id", "modis_rows", "viirs_cd", "viirs_rows", "p_viirs", "pred_scaled", "split"]] \
    .rename(columns={"viirs_cd": "viirs_present", "viirs_rows": "viirs_raw_rows"}) \
    .to_csv(OUT / "cell_day_algeria.csv", index=False)

raw_m = M.assign(sensor="MODIS")[["acq_date", "acq_time", "latitude", "longitude", "sensor", "satellite", "frp", "confidence"]]
raw_v = V.assign(sensor="VIIRS")[["acq_date", "acq_time", "latitude", "longitude", "sensor", "satellite", "frp", "confidence"]]
pd.concat([raw_m, raw_v]).to_csv(OUT / "raw_detections_algeria.csv.gz", index=False)

cells = cd[["cy", "cx", "cell_id"]].drop_duplicates()
cells.assign(lat_min=cells.cy * CELL, lat_max=(cells.cy + 1) * CELL,
             lon_min=cells.cx * CELL, lon_max=(cells.cx + 1) * CELL)[
    ["cell_id", "lat_min", "lat_max", "lon_min", "lon_max"]].to_csv(OUT / "grid_cells_algeria.csv", index=False)

json.dump({"split_method": "hidden-years rehearsal: train 2016-2025, test 2013-2015 (VIIRS hidden), excluded months: " + ", ".join(OUTAGE),
           "metrics": {k: v for k, v in ev.items()}}, open(OUT / "evaluation.json", "w"), indent=2, default=float)
json.dump({"model": "HistGradientBoostingClassifier (scikit-learn)", "version": "v1-" + str(pd.Timestamp.today().date()),
           "target": "probability that VIIRS has >=1 vegetation-fire detection (confidence n/h, type 0) in the same cell on the same UTC day",
           "grid": {"cell_degrees": CELL, "cell_id": "<floor(lat/0.01)>_<floor(lon/0.01)>", "region_crop": crop},
           "time_matching": "same UTC calendar day, same 0.01 degree cell",
           "modis_filter": "confidence >= 30, type == 0", "viirs_filter": "confidence in n,h; type == 0",
           "training_period_final": "2013-01 to 2025-12 (excluding the listed gap months)",
           "features": FE, "linkage_scale_final": float(scale_fin),
           "notes": "pred_scaled = p_viirs * linkage_scale; the scale assumes a constant VIIRS/MODIS-linked ratio, which our drift results contradict",
           "predicted_for": "cell-days with a MODIS detection only; VIIRS-only cell-days are not predicted"},
          open(OUT / "model_metadata.json", "w"), indent=2)
print("wrote:", [p.name for p in OUT.iterdir()])

53,126 MODIS cell-days | 56,425 VIIRS cell-days | 7,410 of them also have MODIS the same day


,value
cell-day: AUC,0.691
cell-day: Brier,0.200
cell-day: Brier of a constant guess,0.222
linkage scale (train years),7.549


,median monthly err (%),total bias (%)
"monthly, linked part only (sum p vs VIIRS on MODIS cell-days)",NaN,-5.9
"monthly, scaled to all VIIRS cell-days",67.7,-8.7


rows per split: {'train': 50625, 'reconstructed': 30156, 'test_hidden': 20702, 'viirs_gap': 2236} | final linkage scale: 7.61
wrote: ['cell_day_algeria.csv', 'evaluation.json', 'grid_cells_algeria.csv', 'model_metadata.json', 'raw_detections_algeria.csv.gz']


In [30]:
import glob
def schema(path, **kw):
    try:
        df = pd.read_parquet(path) if str(path).endswith(".parquet") else pd.read_csv(path, nrows=1000, **kw)
        return {c: str(t) for c, t in df.dtypes.items()}
    except Exception as e:
        return {"ERROR": str(e)[:60]}

files = {
 "NASA raw MODIS":           RAW / "modis_archive.csv",
 "NASA raw VIIRS":           RAW / "viirs_archive.csv",
 "grid MODIS (README table)": PROC / "algeria_grid_m.parquet",
 "grid VIIRS (README table)": PROC / "algeria_grid_v.parquet",
 "our cell-day CSV":         PROC / "export_algeria" / "cell_day_algeria.csv",
 "our reconstruction":       PROC / "algeria_harmonized_2003_2012.parquet",
}
for name, p in files.items():
    print(f"\n== {name}  ({Path(p).name})")
    print(schema(p))


== NASA raw MODIS  (modis_archive.csv)
{'latitude': 'float64', 'longitude': 'float64', 'brightness': 'float64', 'scan': 'float64', 'track': 'float64', 'acq_date': 'str', 'acq_time': 'int64', 'satellite': 'str', 'instrument': 'str', 'confidence': 'int64', 'version': 'float64', 'bright_t31': 'float64', 'frp': 'float64', 'daynight': 'str', 'type': 'int64'}

== NASA raw VIIRS  (viirs_archive.csv)
{'latitude': 'float64', 'longitude': 'float64', 'brightness': 'float64', 'scan': 'float64', 'track': 'float64', 'acq_date': 'str', 'acq_time': 'int64', 'satellite': 'str', 'instrument': 'str', 'confidence': 'str', 'version': 'int64', 'bright_t31': 'float64', 'frp': 'float64', 'daynight': 'str', 'type': 'int64'}

== grid MODIS (README table)  (algeria_grid_m.parquet)
{'lat': 'float64', 'lon': 'float64', 'year': 'int32', 'month': 'int32', 'count': 'int64'}

== grid VIIRS (README table)  (algeria_grid_v.parquet)
{'lat': 'float64', 'lon': 'float64', 'year': 'int32', 'month': 'int32', 'count': 'int64'

In [31]:
from scipy.spatial import cKDTree

RAWD = Path("../data/raw") / "algeria"
VCOLS = ["latitude", "longitude", "brightness", "scan", "track", "acq_date", "acq_time", "satellite",
         "instrument", "confidence", "version", "bright_t31", "frp", "daynight", "type"]
crop = CROP["algeria"]
KX, KY = 111.32 * np.cos(np.radians(35.6)), 110.57

def rd(fname, keep, d0, d1):
    parts = []
    for ch in pd.read_csv(RAWD / fname, usecols=VCOLS, chunksize=500_000):
        ch = ch[ch.latitude.between(*crop[:2]) & ch.longitude.between(*crop[2:])]
        ch = ch[ch.acq_date.between(d0, d1)]
        parts.append(ch[keep(ch)])
    return pd.concat(parts, ignore_index=True)

V = rd("viirs_archive.csv", lambda c: c.confidence.isin(["n", "h"]) & (c.type == 0), "2013-01-01", "2026-12-31")
M = rd("modis_archive.csv", lambda c: (c.confidence.astype(float) >= 30) & (c.type == 0), "2003-01-01", "2025-12-31")
dM, dV = pd.to_datetime(M.acq_date), pd.to_datetime(V.acq_date)
M["t"], V["t"] = dM.dt.year * 12 + dM.dt.month - 1, dV.dt.year * 12 + dV.dt.month - 1

def feats(df):
    d = pd.to_datetime(df.acq_date)
    return pd.DataFrame({"brightness": df.brightness, "scan": df.scan, "track": df.track, "frp": df.frp,
                         "conf": df.confidence.astype(float), "aqua": (df.satellite == "Aqua").astype(int),
                         "day": (df.daynight == "D").astype(int), "hour": df.acq_time // 100, "month": d.dt.month})

def xyz(df, d):
    return np.column_stack([df.longitude * KX, df.latitude * KY, (d - D0).dt.days.to_numpy() * 10_000.0])

# 1. training rows: MODIS 2013+, excluding the gap months. Target: VIIRS rows within 0.75 km on the same day.
trm = (M.acq_date >= "2013-01-01") & ~M.t.isin(out_t)
n = cKDTree(xyz(V, dV)).query_ball_point(xyz(M[trm], dM[trm]), 0.75, return_length=True, workers=-1)
mdl = HistGradientBoostingRegressor(loss="poisson", max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                    min_samples_leaf=50, l2_regularization=1.0, random_state=0).fit(feats(M[trm]), n)
Vk = V[~V.t.isin(out_t)]
scale = len(Vk) / n.sum()                                    # VIIRS rows per linked MODIS row (training years)

# 2. expected VIIRS rows per MODIS row for 2003-2012
Mp = M[M.acq_date < "2013-01-01"].copy()
lam = mdl.predict(feats(Mp)) * scale
if "mt_pre" in globals():                                    # Cell 5 was run: force each month's total to the hybrid's curved total
    cdays = len(Vk.assign(cy=np.floor(Vk.latitude / CELL), cx=np.floor(Vk.longitude / CELL)).drop_duplicates(["cy", "cx", "acq_date"]))
    target = mt_pre.total_curved * (len(Vk) / cdays)         # cell-days -> rows
    summ = pd.Series(lam, index=Mp.index).groupby(Mp.t).sum()
    fac = (target / summ).replace([np.inf, -np.inf], np.nan).reindex(Mp.t).to_numpy()
    lam = lam * np.where(np.isnan(fac), 1.0, fac)
    print("monthly totals rescaled to the hybrid (Cell 5)")
else:
    print("Cell 5 not run: using the row model's own totals (less accurate)")

# 3. stochastic rounding: k rows per MODIS row, average = lam, so monthly sums keep the expected total
rng = np.random.default_rng(0)
k = np.floor(lam).astype(int) + (rng.random(len(lam)) < lam - np.floor(lam))
est = Mp.loc[Mp.index.repeat(k)].copy()
est["frp"] = est.frp * (V.frp.median() / M[trm].frp.median())   # rough
for c in ["brightness", "scan", "track", "bright_t31", "version"]:
    est[c] = np.nan
est["confidence"], est["type"] = "n", 0
est["satellite"], est["instrument"] = V.satellite.mode()[0], "VIIRS_EST"

# 4. same 15 columns as the NASA VIIRS file, real rows from 2013
out = pd.concat([est[VCOLS], V[VCOLS]]).sort_values(["acq_date", "acq_time"]).reset_index(drop=True)
out.to_csv(PROC / "viirs_harmonized_algeria.csv", index=False)

raw_cols = list(pd.read_csv(RAWD / "viirs_archive.csv", nrows=1).columns)
print("columns identical to the NASA VIIRS file:", list(out.columns) == raw_cols)
yr = out.assign(year=out.acq_date.str[:4], src=np.where(out.instrument == "VIIRS_EST", "estimated", "real")) \
        .groupby(["year", "src"]).size().unstack(fill_value=0)
display(yr)
print("gap months that still hold real rows (check with your binome):", OUTAGE)

Cell 5 not run: using the row model's own totals (less accurate)
columns identical to the NASA VIIRS file: True


src,estimated,real
year,,
2003,7822,0
2004,7830,0
2005,10713,0
2006,7581,0
2007,23471,0
2008,8726,0
2009,11908,0
2010,8931,0
2011,6362,0


gap months that still hold real rows (check with your binome): ['2017-12', '2018-10', '2022-04', '2022-07', '2022-08', '2024-01', '2024-02', '2024-06', '2024-07', '2024-11', '2025-12', '2026-01', '2026-03', '2026-04', '2026-07']


In [32]:
print(pd.read_csv(RAWD / "viirs_archive.csv", usecols=["acq_date"]).acq_date.min())

2012-07-04


In [33]:
pq_final = fit_quad_smear(tot)
pl_final = fit_loglog_smear(tot)
mt_pre = pd.DataFrame({"modis": tot_pre.astype(float)}).rename_axis("t")
mt_pre["year"], mt_pre["month"] = mt_pre.index // 12, mt_pre.index % 12 + 1
mt_pre["total_curved"]   = np.clip(pred_quad_smear(pq_final, mt_pre.modis), 0, None)
mt_pre["total_straight"] = np.clip(pred_loglog_smear(pl_final, mt_pre.modis), 0, None)
mt_pre["outside_range"]  = (mt_pre.modis < tot.modis.min()) | (mt_pre.modis > tot.modis.max())
display(mt_pre[mt_pre.outside_range][["year", "month", "modis", "total_curved", "total_straight"]].round(0))
print(mt_pre.groupby("year")[["total_curved", "total_straight"]].sum().round(0))

,year,month,modis,total_curved,total_straight
t,,,,,
24037,2003,2,0.0,27.0,17.0
24091,2007,8,3059.0,5179.0,3562.0
24107,2008,12,0.0,27.0,17.0
24108,2009,1,0.0,27.0,17.0
24151,2012,8,5351.0,8657.0,5157.0


      total_curved  total_straight
year                              
2003        4185.0          3915.0
2004        5313.0          5045.0
2005        6376.0          5962.0
2006        5828.0          5537.0
2007        8408.0          6666.0
2008        5098.0          4755.0
2009        6300.0          5610.0
2010        6246.0          5894.0
2011        4639.0          4415.0
2012       13731.0          9915.0


In [34]:
print(pd.read_csv(RAWD / "viirs_archive.csv", usecols=["acq_date"]).acq_date.min())

2012-07-04


In [35]:
from scipy.spatial import cKDTree

RAWD = Path("../data/raw") / "algeria"
VCOLS = ["latitude", "longitude", "brightness", "scan", "track", "acq_date", "acq_time", "satellite",
         "instrument", "confidence", "version", "bright_t31", "frp", "daynight", "type"]
crop, FIRST_V, D0 = CROP["algeria"], "2012-07-04", pd.Timestamp("2013-01-01")
JUL12 = 2012 * 12 + 6                                   # month index of July 2012
KX, KY = 111.32 * np.cos(np.radians(35.6)), 110.57

def rd(fname, keep, d0, d1):
    parts = []
    for ch in pd.read_csv(RAWD / fname, usecols=VCOLS, chunksize=500_000):
        ch = ch[ch.latitude.between(*crop[:2]) & ch.longitude.between(*crop[2:])]
        ch = ch[ch.acq_date.between(d0, d1)]
        parts.append(ch[keep(ch)])
    return pd.concat(parts, ignore_index=True)

V = rd("viirs_archive.csv", lambda c: c.confidence.isin(["n", "h"]) & (c.type == 0), FIRST_V, "2026-12-31")
M = rd("modis_archive.csv", lambda c: (c.confidence.astype(float) >= 30) & (c.type == 0), "2003-01-01", "2025-12-31")
dM, dV = pd.to_datetime(M.acq_date), pd.to_datetime(V.acq_date)
M["t"], V["t"] = dM.dt.year * 12 + dM.dt.month - 1, dV.dt.year * 12 + dV.dt.month - 1

# ---- free test: real VIIRS cell-days for Jul-Dec 2012 vs what the two curves predicted for those months ----
Vcd = V.assign(cy=np.floor(V.latitude / CELL), cx=np.floor(V.longitude / CELL)).drop_duplicates(["cy", "cx", "acq_date"])
real = Vcd[Vcd.t.between(JUL12, JUL12 + 5)].groupby("t").size()
chk = pd.DataFrame({"MODIS cell-days": mt_pre.modis.reindex(real.index), "real VIIRS": real,
                    "curved": mt_pre.total_curved.reindex(real.index), "straight": mt_pre.total_straight.reindex(real.index)})
chk.index = [f"2012-{t % 12 + 1:02d}" for t in chk.index]
display(chk.round(0))                                    # July is partial (starts on the 4th): ignore its comparison

# ---- the row model, trained from the first real VIIRS day ----
def feats(df):
    d = pd.to_datetime(df.acq_date)
    return pd.DataFrame({"brightness": df.brightness, "scan": df.scan, "track": df.track, "frp": df.frp,
                         "conf": df.confidence.astype(float), "aqua": (df.satellite == "Aqua").astype(int),
                         "day": (df.daynight == "D").astype(int), "hour": df.acq_time // 100, "month": d.dt.month})
def xyz(df, d):
    return np.column_stack([df.longitude * KX, df.latitude * KY, (d - D0).dt.days.to_numpy() * 10_000.0])

trm = (M.acq_date >= FIRST_V) & ~M.t.isin(out_t)
n = cKDTree(xyz(V, dV)).query_ball_point(xyz(M[trm], dM[trm]), 0.75, return_length=True, workers=-1)
mdl = HistGradientBoostingRegressor(loss="poisson", max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                    min_samples_leaf=50, l2_regularization=1.0, random_state=0).fit(feats(M[trm]), n)
Vk = V[~V.t.isin(out_t)]
scale = len(Vk) / n.sum()
Vkcd = Vk.assign(cy=np.floor(Vk.latitude / CELL), cx=np.floor(Vk.longitude / CELL)).drop_duplicates(["cy", "cx", "acq_date"])
rpc = len(Vk) / len(Vkcd)                                # VIIRS rows per cell-day
print(f"rows per cell-day: {rpc:.2f}")

# ---- expected rows per MODIS row before the first real day, then force each month to the hybrid's curved total ----
Mp = M[M.acq_date < FIRST_V].copy()
lam = mdl.predict(feats(Mp)) * scale
target = mt_pre.total_curved * rpc                       # hybrid monthly totals, in rows
summ = pd.Series(lam, index=Mp.index).groupby(Mp.t).sum()
fac = (target.reindex(summ.index) / summ).replace([np.inf, -np.inf], np.nan)
fac[fac.index >= JUL12] = 1.0                            # July 2012: only days 1-3 are estimated, the rest is real
lam = lam * fac.reindex(Mp.t).fillna(1.0).to_numpy()
display(fac.groupby(fac.index // 12).median().round(2).to_frame("median factor (hybrid total / row-model total)"))

rng = np.random.default_rng(0)
k = np.floor(lam).astype(int) + (rng.random(len(lam)) < lam - np.floor(lam))
est = Mp.loc[Mp.index.repeat(k)].copy()
est["frp"] = est.frp * (V.frp.median() / M[trm].frp.median())
for c in ["brightness", "scan", "track", "bright_t31", "version"]:
    est[c] = np.nan
est["confidence"], est["type"] = "n", 0
est["satellite"], est["instrument"] = V.satellite.mode()[0], "VIIRS_EST"

# ---- the file the app reads: same 15 columns as the NASA VIIRS file ----
out = pd.concat([est[VCOLS], V[VCOLS]]).sort_values(["acq_date", "acq_time"]).reset_index(drop=True)
out.to_csv(PROC / "viirs_harmonized_algeria.csv", index=False)
print("columns identical to the NASA VIIRS file:", list(out.columns) == list(pd.read_csv(RAWD / "viirs_archive.csv", nrows=1).columns))
display(out.assign(year=out.acq_date.str[:4], src=np.where(out.instrument == "VIIRS_EST", "estimated", "real"))
           .groupby(["year", "src"]).size().unstack(fill_value=0))
pre = target[target.index < JUL12]
display(pd.DataFrame({"rows in file": est[est.t < JUL12].groupby(est.t // 12).size(),
                      "hybrid target (rows)": pre.groupby(pre.index // 12).sum().round(0)}))

# ---- side files for the app: monthly and yearly totals with their uncertainty and flags (not in the main file) ----
mm = cube.loc[cube.year <= 2015].groupby("t")[["viirs", "hyb_q"]].sum()
mm = mm[mm.viirs > 0]
q10, q90 = (mm.hyb_q / mm.viirs).quantile([0.1, 0.9])
am = tot[tot.viirs > 0].copy(); am["year"] = am.index // 12; am["pred"] = pred_quad_smear(pq, am.modis)
yb = 100 * (am.groupby("year").pred.sum() / am.groupby("year").viirs.sum() - 1)
mb, sb = yb.mean(), yb.std()
side = mt_pre.reset_index()[["t", "year", "month", "modis", "total_curved", "total_straight", "outside_range"]].copy()
side["rows_curved"] = side.total_curved * rpc
side["low_month"], side["high_month"] = side.total_curved / q90, side.total_curved / q10
side["low_confidence"] = side.outside_range | (side.modis == 0)
side["source"] = "reconstructed"
side.drop(columns="outside_range").to_csv(PROC / "viirs_harmonized_algeria_monthly.csv", index=False)
yearly = side.groupby("year")[["total_curved", "total_straight"]].sum()
yearly["low_band"] = yearly.total_curved / (1 + (mb + 2 * sb) / 100)
yearly["high_band"] = yearly.total_curved / (1 + (mb - 2 * sb) / 100)
yearly["has_low_confidence_month"] = side.groupby("year").low_confidence.any()
yearly.round(0).to_csv(PROC / "viirs_harmonized_algeria_yearly.csv")
display(yearly.round(0))

,MODIS cell-days,real VIIRS,curved,straight
2012-07,898.0,1679,1798.0,1584.0
2012-08,5351.0,8663,8657.0,5157.0
2012-09,756.0,1388,1562.0,1413.0
2012-10,190.0,569,539.0,568.0
2012-11,8.0,33,77.0,74.0
2012-12,5.0,55,62.0,57.0


rows per cell-day: 1.67


,median factor (hybrid total / row-model total)
t,
2003,19.78
2004,19.27
2005,5.37
2006,6.12
2007,14.33
2008,8.81
2009,6.65
2010,10.94
2011,14.16


columns identical to the NASA VIIRS file: True


src,estimated,real
year,,
2003,6920,0
2004,8851,0
2005,10623,0
2006,9699,0
2007,14003,0
2008,8443,0
2009,10441,0
2010,10424,0
2011,7731,0


ValueError: cannot reindex on an axis with duplicate labels

In [36]:
f = pd.read_csv(PROC / "viirs_harmonized_algeria.csv", usecols=["latitude", "longitude", "instrument", "acq_date"])
f["src"] = np.where(f.instrument == "VIIRS_EST", "estimated", "real")
f["lat_band"] = pd.cut(f.latitude, [34.0, 35.0, 35.5, 36.0, 36.5, 37.2])
display(pd.crosstab(f.lat_band, f.src, normalize="columns").mul(100).round(1))
print("share of estimated rows on a duplicate coordinate:",
      round(100 * f[f.src == "estimated"].duplicated(["latitude", "longitude", "acq_date"], keep=False).mean(), 1), "%",
      "| real:", round(100 * f[f.src == "real"].duplicated(["latitude", "longitude", "acq_date"], keep=False).mean(), 1), "%")

src,estimated,real
lat_band,,
"(34.0, 35.0]",8.2,7.2
"(35.0, 35.5]",10.5,8.6
"(35.5, 36.0]",8.6,9.3
"(36.0, 36.5]",30.6,27.1
"(36.5, 37.2]",42.1,47.7


share of estimated rows on a duplicate coordinate: 95.8 % | real: 0.0 %


In [37]:
CELL = 0.01
CROP = {"algeria": (34.0, 37.2, -2.3, 8.7)}
CROP["usa"] = (32.5, 42.0, -124.5, -114.0)        # EDIT: your USA box (lat_min, lat_max, lon_min, lon_max)
REGS = ["algeria", "portugal", "zambia", "usa"]   # remove "usa" until the files exist

def cd_monthly(reg):
    d, crop, out = Path("../data/raw") / reg, CROP.get(reg), {}
    for sensor, fname, keep in [
        ("modis", "modis_archive.csv", lambda c: (c.confidence.astype(float) >= 30) & (c.type == 0)),
        ("viirs", "viirs_archive.csv", lambda c: c.confidence.isin(["n", "h"]) & (c.type == 0))]:
        parts = []
        for ch in pd.read_csv(d / fname, usecols=["latitude", "longitude", "acq_date", "confidence", "type"], chunksize=500_000):
            if crop:
                ch = ch[ch.latitude.between(*crop[:2]) & ch.longitude.between(*crop[2:])]
            ch = ch[ch.acq_date.between("2013-01-01", "2025-12-31")]
            ch = ch[keep(ch)]
            parts.append(pd.DataFrame({"cy": np.floor(ch.latitude / CELL).astype("int32"),
                                       "cx": np.floor(ch.longitude / CELL).astype("int32"),
                                       "date": ch.acq_date}).drop_duplicates())
        u = pd.concat(parts).drop_duplicates()
        t = pd.to_datetime(u.date)
        u["t"] = t.dt.year * 12 + t.dt.month - 1
        out[sensor] = u.groupby("t").size()
    df = pd.DataFrame(out).reindex(range(2013 * 12, 2026 * 12)).fillna(0)
    if reg == "algeria":
        df = df.drop(index=out_t, errors="ignore")
    print(reg, len(df), "months |", int(df.modis.sum()), "MODIS cell-days |", int(df.viirs.sum()), "VIIRS cell-days")
    return df

ALL = pd.concat([cd_monthly(r).assign(region=r) for r in REGS]).rename_axis("t").reset_index()
ALL = ALL[ALL.viirs > 0].copy()
ALL["year"], ALL["x"], ALL["y"] = ALL.t // 12, np.log(ALL.modis + 1), np.log(ALL.viirs + 1)

def design(df, mode):
    if mode == "intercepts":
        return np.column_stack([(df.region == r) * 1.0 for r in REGS] + [df.x])
    return np.column_stack([np.ones(len(df)), df.x])

def fit_predict(trn, tst, mode):
    A = design(trn, mode)
    coef = np.linalg.lstsq(A, trn.y, rcond=None)[0]
    smear = np.mean(np.exp(trn.y - A @ coef))
    return smear * np.exp(design(tst, mode) @ coef) - 1

def summarize(rows):
    r = pd.concat(rows)
    g = r.groupby(["region", "setup"])
    return pd.DataFrame({"median monthly err (%)": g.apply(lambda d: 100 * np.median(np.abs(d.viirs - d.pred) / d.viirs)),
                         "total bias (%)": g.apply(lambda d: 100 * (d.pred.sum() / d.viirs.sum() - 1))}).round(1)

# 1. leave-one-year-out inside each region
rows = []
for r in REGS:
    for yr in ALL[ALL.region == r].year.unique():
        te = ALL[(ALL.region == r) & (ALL.year == yr)]
        for mode in ("own", "pooled", "intercepts"):
            trn = ALL[(ALL.region == r) & (ALL.year != yr)] if mode == "own" else ALL[~((ALL.region == r) & (ALL.year == yr))]
            rows.append(te.assign(setup=mode, pred=fit_predict(trn, te, mode))[["region", "setup", "viirs", "pred"]])
display(summarize(rows).unstack("setup"))

# 2. leave-one-region-out: the model never saw this region
rows = []
for r in REGS:
    te, trn = ALL[ALL.region == r], ALL[ALL.region != r]
    rows.append(te.assign(setup="unseen region", pred=fit_predict(trn, te, "pooled"))[["region", "setup", "viirs", "pred"]])
display(summarize(rows))

algeria 145 months | 22312 MODIS cell-days | 56425 VIIRS cell-days
portugal 156 months | 28374 MODIS cell-days | 41530 VIIRS cell-days
zambia 156 months | 2349469 MODIS cell-days | 6023854 VIIRS cell-days
usa 156 months | 158635 MODIS cell-days | 300906 VIIRS cell-days


median monthly err (%)              total bias (%)            
setup                intercepts   own pooled     intercepts  own pooled
region                                                                 
algeria                    28.3  14.2   32.8           39.9 -3.1   23.1
portugal                   23.7  22.8   98.2            4.4 -1.1  110.8
usa                        13.6  12.0   41.8           -5.3 -7.9   37.9
zambia                     16.0  13.9   27.8          -11.0  1.8  -26.4

,,median monthly err (%),total bias (%)
region,setup,,
algeria,unseen region,34.5,11.9
portugal,unseen region,130.9,134.7
usa,unseen region,57.5,55.3
zambia,unseen region,59.0,-66.8


In [38]:
rng = np.random.default_rng(1)
m = out.instrument == "VIIRS_EST"
out.loc[m, "latitude"]  += rng.uniform(-0.005, 0.005, m.sum())
out.loc[m, "longitude"] += rng.uniform(-0.005, 0.005, m.sum())
out.to_csv(PROC / "viirs_harmonized_algeria.csv", index=False)
print("duplicates now:", round(100 * out[m].duplicated(["latitude", "longitude", "acq_date"], keep=False).mean(), 2), "%")

duplicates now: 0.0 %
